## 检查是否用的是本地的pip

In [2]:
import sys
print(sys.executable)

D:\Year2Sem2\Anaconda\python.exe


In [3]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

In [4]:
!nvidia-smi

Thu Jun  4 02:59:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.47                 KMD Version: 610.47        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   55C    P0             10W /   35W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
import torch
print("CUDA 是否可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("当前显卡名称:", torch.cuda.get_device_name(0))
    # 尝试创建 an 简单的 Tensor 并推送到 GPU
    try:
        x = torch.randn(1, 3, 224, 224).cuda()
        print("基础显卡测试成功！")
    except Exception as e:
        print("基础显卡测试失败，错误为:", e)

CUDA 是否可用: True
当前显卡名称: NVIDIA GeForce RTX 3050 4GB Laptop GPU
基础显卡测试成功！


## 加载模型，生成一句话
### Transformer中有哪些层：
以distilgpt2(decoder-only)为例，每个Decoder Layer包含两子层，
    Self Attention(GPT2Attention):计算序列中每个token对于其他token的注意力，融合全局信息
    前馈网络层（GPT2MLP）:两个线性变换+激活函数（ReLU）,对每个token独立做非线性变换，加上输入端的Token+Position Embedding、每层的LayerNorm和最后的 输出线性头（Im_head）构成完整的推理计算流
With torch.no_grad(): 告诉PyTorch不要构建计算图，不计算梯度，节省显存和算力。

### DistilGPT2 就是一个 decoder‑only 模型。它的每一层（Decoder Layer）由两个主要积木组成：

1. Masked 自注意力（GPT2Attention）
与完整自注意力一样，但多了一个 因果掩码：不允许当前词看到它后面的词（只能看到自身及之前的内容）。
做生成时，我们总是根据已有的前缀预测下一个词，这一点很关键。

2. 前馈网络（GPT2MLP）
一个简单的两层全连接网络，中间有激活函数（GELU）。可以对每个位置的向量进行更复杂的非线性变换。

这两个积木之间都有 残差连接（把输入直接加到输出上）和 层归一化（保持数值稳定），帮助训练深层网络。

### 一个完整的推理流程是：

输入 prompt → Tokenizer 拆成数字 ID → 查表得到 token embedding → 加上位置 embedding

送入 N 层（DistilGPT2 是 6 层）decoder layer，逐层加工

最后一层的输出经过 LayerNorm 和一个线性层（lm_head）投影到词汇表大小，得到 logits

取最后一个位置的 logits，用 softmax 变成概率，然后采样出一个新 token

把新 token 拼到序列末尾，重复第 2–4 步，直到生成完

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()

tokenizer.pad_token = tokenizer.eos_token

prompt = "The future of AI is"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(
    **inputs,
    max_new_tokens=20,
    do_sample=True,
    temperature=0.9,
    pad_token_id=tokenizer.eos_token_id
)
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)



The future of AI is a question as to who has the best technical capabilities, but I do feel that a lot of people


## Transformer 的每一层（Decoder Layer）内部包含 自注意力（Self-Attention） 和 前馈网络（FFN）。我们通过 注册 forward hook 来捕获每层的输入、输出张量的 shape。

In [24]:
# 定义一个hook函数专门打印 shape
def print_shape_hook(module, input, output):
    # input是 tuple, 第一个元素是hidden_states
    inp = input[0]
    out = output[0]
    print(f"{module.__class__.__name__}:input {tuple(inp.shape)} -> output {tuple(out.shape)}")

# 对所有的 decoder layer子模块注册 hook
hooks = []
for name, module in model.named_modules():
    if "transformer.h." in name and name.count('.') == 2: #只对decoder layer (如h.0, h.1)整体注册
        hooks.append(module.register_forward_hook(print_shape_hook))

# 现在再做一次前向传播，会打印每一层输入/输出的shape
with torch.no_grad():
    model(**inputs) # prompt编码后 input_ids直接送入模型

# 移除hooks,避免影响后续运行
for hook in hooks:
    hook.remove()
        

GPT2Block:input (1, 5, 768) -> output (1, 5, 768)
GPT2Block:input (1, 5, 768) -> output (1, 5, 768)
GPT2Block:input (1, 5, 768) -> output (1, 5, 768)
GPT2Block:input (1, 5, 768) -> output (1, 5, 768)
GPT2Block:input (1, 5, 768) -> output (1, 5, 768)
GPT2Block:input (1, 5, 768) -> output (1, 5, 768)
